# LSS Step Through

这个 notebook 用来逐行跑通 `learn2027/src` 里的 LSS 源码。

学习目标不是一口气训练模型，而是看清楚每一步：

```text
数据进来是什么 shape
代码做了什么变换
为什么 LSS 需要这个变换
变换后数据变成什么样子
```

In [ ]:
from pathlib import Path
import os
import sys

def is_repo_root(path):
    return (path / "learn2027" / "src" / "models.py").exists()

def find_repo_root():
    # 1. Best case: Jupyter was started from the repo or a child directory.
    for path in [Path.cwd(), *Path.cwd().parents]:
        if is_repo_root(path):
            return path

    # 2. Environment variable fallback, useful when VS Code starts notebooks
    #    from a directory outside this repository.
    env_repo_root = os.environ.get("LSS_REPO_ROOT")
    if env_repo_root:
        repo_root = Path(env_repo_root).expanduser()
        if is_repo_root(repo_root):
            return repo_root

    # 3. Optional local file config. The real config file should stay outside
    #    Git or be ignored by Git.
    try:
        import importlib.util
        config_candidates = []
        if os.environ.get("LSS_LOCAL_CONFIG"):
            config_candidates.append(Path(os.environ["LSS_LOCAL_CONFIG"]).expanduser())
        config_candidates.append(Path.home() / ".config" / "lss2027" / "local_config.py")
        for path in [Path.cwd(), *Path.cwd().parents]:
            config_candidates.append(path / "learn2027" / "local_config.py")
        for config_path in config_candidates:
            if not config_path.exists():
                continue
            spec = importlib.util.spec_from_file_location("lss_local_config", config_path)
            config = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(config)
            repo_root = Path(config.REPO_ROOT).expanduser()
            if is_repo_root(repo_root):
                return repo_root
    except Exception as exc:
        print("Ignoring invalid local_config.py:", exc)

    raise FileNotFoundError(
        "Could not find the lift-splat-shoot repo root. "
        "Start Jupyter from the repo root, set LSS_REPO_ROOT, set "
        "LSS_LOCAL_CONFIG, or create ~/.config/lss2027/local_config.py."
    )

repo_root = find_repo_root()
learn_root = repo_root / "learn2027"
mpl_config = learn_root / ".matplotlib"
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_config)
sys.path.insert(0, str(learn_root))

print("repo_root:", repo_root)
print("using learning src:", learn_root / "src")

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.models import LiftSplatShoot

torch.set_printoptions(precision=3, sci_mode=False)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 1. 建立和源码一致的配置

这些配置来自 `src/train.py`。先不要改它们，先看懂默认设置。

In [ ]:
grid_conf = {
    "xbound": [-50.0, 50.0, 0.5],
    "ybound": [-50.0, 50.0, 0.5],
    "zbound": [-10.0, 10.0, 20.0],
    "dbound": [4.0, 45.0, 1.0],
}

data_aug_conf = {
    "resize_lim": (0.193, 0.225),
    "final_dim": (128, 352),
    "rot_lim": (-5.4, 5.4),
    "H": 900,
    "W": 1600,
    "rand_flip": True,
    "bot_pct_lim": (0.0, 0.22),
    "cams": [
        "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
        "CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT",
    ],
    "Ncams": 5,
}

print(grid_conf)
print(data_aug_conf)

## 2. 先只实例化模型

`LiftSplatShoot.__init__` 会创建：

```text
dx / bx / nx
frustum
CamEncode
BevEncode
```

第一次运行这里可能会下载 EfficientNet 预训练权重。

In [ ]:
model = LiftSplatShoot(grid_conf, data_aug_conf, outC=1)
model.eval()

print("dx:", model.dx)
print("bx:", model.bx)
print("nx:", model.nx)
print("frustum shape:", tuple(model.frustum.shape))
print("D:", model.D)
print("camC:", model.camC)

## 3. 可视化 `create_frustum()` 的结果

`frustum[d, h, w] = [image_x, image_y, depth]`

它的目的：给每个图像特征点准备多个候选深度。

In [ ]:
frustum = model.frustum.detach()
print("frustum shape:", tuple(frustum.shape))
print("first point:", frustum[0, 0, 0])
print("last point:", frustum[-1, -1, -1])

D, fH, fW, _ = frustum.shape
depth_values = frustum[:, 0, 0, 2]
print("depth values:", depth_values[:10], "...", depth_values[-5:])

plt.figure(figsize=(6, 3))
plt.plot(depth_values.numpy(), marker="o")
plt.title("Depth candidates used by create_frustum")
plt.xlabel("depth index")
plt.ylabel("meters")
plt.grid(True)
plt.show()

In [ ]:
d_index = 0
xy = frustum[d_index, :, :, :2].reshape(-1, 2)

plt.figure(figsize=(8, 3))
plt.scatter(xy[:, 0], xy[:, 1], s=30)
plt.title(f"Image-plane feature locations at depth={frustum[d_index,0,0,2].item():.1f}m")
plt.xlim(0, data_aug_conf["final_dim"][1])
plt.ylim(data_aug_conf["final_dim"][0], 0)
plt.xlabel("image x")
plt.ylabel("image y")
plt.grid(True)
plt.show()

## 4. 下一步

下一节建议进入：

```python
CamEncode.get_depth_feat()
```

先观察为什么 `depthnet` 输出 `D + C` 个通道。